# 04 · SMOTE-NC: selección de k

Genera registros de la clase minoritaria por interpolación entre vecinos (SMOTE-NC si hay variables categóricas, SMOTE si no) hasta la proporción 1:1, y elige `k_neighbors` ∈ {3, 5, 7} por el AUC-PR en validación de una regresión logística (proxy).

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import Counter
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)
from utils.config_loader import load_config

config = load_config("dataset_config.yaml")
RANDOM_STATE = config.get("random_state", 42)
print("Dataset:", config["dataset_name"])
USE_SMOTE_NC = len(config.get("categorical_features", [])) > 0
K_GRID = [3, 5, 7]

## Datos (escalados, entrenamiento interno y validación)

In [ ]:
SPLIT_DIR = "outputs/splits"
X_train = pd.read_csv(f"{SPLIT_DIR}/X_train_in.csv")
X_val = pd.read_csv(f"{SPLIT_DIR}/X_val_in.csv")
y_train = pd.read_csv(f"{SPLIT_DIR}/y_train_in.csv")["y"].values
y_val = pd.read_csv(f"{SPLIT_DIR}/y_val_in.csv")["y"].values

counter = Counter(y_train)
minority = min(counter, key=counter.get)
majority = max(counter, key=counter.get)
sampling_dict = {minority: counter[majority]}
cat_idx = [X_train.columns.get_loc(c) for c in config["categorical_features"]] if USE_SMOTE_NC else []
print("Método:", "SMOTE-NC" if USE_SMOTE_NC else "SMOTE")

## Búsqueda de k

In [ ]:
best_score, best_cfg, best_data = -np.inf, None, None

for k in K_GRID:
    if USE_SMOTE_NC:
        sampler = SMOTENC(categorical_features=cat_idx, k_neighbors=k,
                          sampling_strategy=sampling_dict, random_state=RANDOM_STATE)
    else:
        sampler = SMOTE(k_neighbors=k, sampling_strategy=sampling_dict, random_state=RANDOM_STATE)
    X_res, y_res = sampler.fit_resample(X_train, y_train)

    proxy = LogisticRegression(max_iter=3000, solver="lbfgs")
    proxy.fit(X_res, y_res)
    auc_pr = average_precision_score(y_val, proxy.predict_proba(X_val)[:, 1])
    print(f"k = {k}: AUC-PR validación = {auc_pr:.4f}")

    if auc_pr > best_score:
        best_score, best_cfg, best_data = auc_pr, {"k_neighbors": k}, (X_res, y_res)

## Guardado de la configuración seleccionada

In [ ]:
OUTPUT_DIR = "outputs/smote"
os.makedirs(OUTPUT_DIR, exist_ok=True)
X_best, y_best = best_data
X_best.to_csv(f"{OUTPUT_DIR}/X_train_smote.csv", index=False)
pd.DataFrame({"y": y_best}).to_csv(f"{OUTPUT_DIR}/y_train_smote.csv", index=False)

with open(f"{OUTPUT_DIR}/best_config.json", "w") as f:
    json.dump({"best_auc_pr": float(best_score), "params": best_cfg,
               "method": "SMOTE-NC" if USE_SMOTE_NC else "SMOTE"}, f, indent=4)
print(f"k seleccionado: {best_cfg['k_neighbors']} (AUC-PR = {best_score:.4f})")